***Part 1: Regression Task (California Housing)***

***3.1 Task 1: Load and Split Dataset***

In [14]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("camnugent/california-housing-prices")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'california-housing-prices' dataset.
Path to dataset files: /kaggle/input/california-housing-prices


***Imports***

In [4]:
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error


In [5]:
import pandas as pd
import os

# Load the dataset from the path downloaded by kagglehub
# The 'path' variable is set in the first code cell.
housing_file_path = os.path.join(path, 'housing.csv')
df = pd.read_csv(housing_file_path)

# Handle missing values in 'total_bedrooms' by imputing with the median
df['total_bedrooms'] = df['total_bedrooms'].fillna(df['total_bedrooms'].median())
print(df.head())

# Perform one-hot encoding on the 'ocean_proximity' column
df = pd.get_dummies(df, columns=['ocean_proximity'], drop_first=False) # drop_first=False to keep all categories

# Separate features (X) and target (y)
# The correct target column name in this dataset is 'median_house_value'.
X = df.drop('median_house_value', axis=1)
y = df['median_house_value']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

   longitude  latitude  housing_median_age  total_rooms  total_bedrooms  \
0    -122.23     37.88                41.0        880.0           129.0   
1    -122.22     37.86                21.0       7099.0          1106.0   
2    -122.24     37.85                52.0       1467.0           190.0   
3    -122.25     37.85                52.0       1274.0           235.0   
4    -122.25     37.85                52.0       1627.0           280.0   

   population  households  median_income  median_house_value ocean_proximity  
0       322.0       126.0         8.3252            452600.0        NEAR BAY  
1      2401.0      1138.0         8.3014            358500.0        NEAR BAY  
2       496.0       177.0         7.2574            352100.0        NEAR BAY  
3       558.0       219.0         5.6431            341300.0        NEAR BAY  
4       565.0       259.0         3.8462            342200.0        NEAR BAY  


***3.2 Task 2: Complete all the Task***

***• Regression Task (California Housing):***

In [6]:
lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)


LinearRegression()

In [15]:
print("=== Baseline Linear Regression ===")
print("Coefficients:", lin_reg.coef_)
print("Intercept:", lin_reg.intercept_)


=== Baseline Linear Regression ===
Coefficients: [-2.68382734e+04 -2.54683520e+04  1.10218508e+03 -6.02150567e+00
  1.02789395e+02 -3.81729064e+01  4.82527528e+01  3.94739752e+04
 -1.89265829e+04 -5.87132390e+04  1.17198490e+05 -2.40632251e+04
 -1.54954428e+04]
Intercept: -2256620.7988545513


In [8]:
y_train_pred = lin_reg.predict(X_train)
y_test_pred = lin_reg.predict(X_test)


In [9]:
print("Train MSE:", mean_squared_error(y_train, y_train_pred))
print("Test  MSE:", mean_squared_error(y_test, y_test_pred))


Train MSE: 4683203783.504252
Test  MSE: 4908476721.156623


In [10]:
alpha_grid = {"alpha": np.logspace(-3, 0, 13)}  # 0.001 … 1


In [11]:
ridge = Ridge(random_state=42)
lasso = Lasso(random_state=42, max_iter=10000)


In [16]:
ridge_cv = GridSearchCV(ridge, alpha_grid, cv=5, scoring="neg_mean_squared_error", n_jobs=-1)
lasso_cv = GridSearchCV(lasso, alpha_grid, cv=5, scoring="neg_mean_squared_error", n_jobs=-1)


In [17]:
ridge_cv.fit(X_train, y_train)
lasso_cv.fit(X_train, y_train)


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 6.263e+12, tolerance: 2.207e+10
  model = cd_fast.enet_coordinate_descent(


GridSearchCV(cv=5, estimator=Lasso(max_iter=10000, random_state=42), n_jobs=-1,
             param_grid={'alpha': array([0.001     , 0.00177828, 0.00316228, 0.00562341, 0.01      ,
       0.01778279, 0.03162278, 0.05623413, 0.1       , 0.17782794,
       0.31622777, 0.56234133, 1.        ])},
             scoring='neg_mean_squared_error')

***In a Jupyter environment, please rerun this cell to show the HTML representation or trust the notebook.***

***On GitHub, the HTML representation is unable to render, please try loading this page with nbviewer.org.***

In [18]:
print("\n=== Hyperparameter Tuning Results ===")
print("Best Ridge alpha:", ridge_cv.best_params_["alpha"])
print("Best Ridge CV MSE:", -ridge_cv.best_score_)
print("Best Lasso alpha:", lasso_cv.best_params_["alpha"])
print("Best Lasso CV MSE:", -lasso_cv.best_score_)



=== Hyperparameter Tuning Results ===
Best Ridge alpha: 0.5623413251903491
Best Ridge CV MSE: 4711027514.741423
Best Lasso alpha: 0.001
Best Lasso CV MSE: 4711122401.897191


In [20]:
best_ridge = ridge_cv.best_estimator_
best_lasso = lasso_cv.best_estimator_


In [21]:
ridge_train_pred = best_ridge.predict(X_train)
ridge_test_pred = best_ridge.predict(X_test)
lasso_train_pred = best_lasso.predict(X_train)
lasso_test_pred = best_lasso.predict(X_test)


In [22]:
print("\n=== Ridge (L2) with best alpha ===")
print("Coefficients:", best_ridge.coef_)
print("Train MSE:", mean_squared_error(y_train, ridge_train_pred))
print("Test  MSE:", mean_squared_error(y_test, ridge_test_pred))



=== Ridge (L2) with best alpha ===
Coefficients: [-2.68511871e+04 -2.54837569e+04  1.10241111e+03 -6.02106544e+00
  1.02864900e+02 -3.81746782e+01  4.81702431e+01  3.94726729e+04
 -1.59652564e+04 -5.57278066e+04  1.05323271e+05 -2.10933055e+04
 -1.25369028e+04]
Train MSE: 4683257028.664875
Test  MSE: 4909265851.261515


In [23]:
print("\n=== Lasso (L1) with best alpha ===")
print("Coefficients:", best_lasso.coef_)
print("Train MSE:", mean_squared_error(y_train, lasso_train_pred))
print("Test  MSE:", mean_squared_error(y_test, lasso_test_pred))



=== Lasso (L1) with best alpha ===
Coefficients: [-2.68382793e+04 -2.54683579e+04  1.10218525e+03 -6.02150576e+00
  1.02789417e+02 -3.81729083e+01  4.82527346e+01  3.94739750e+04
  1.66420878e+04 -2.31445583e+04  1.52763022e+05  1.15054322e+04
  2.00732150e+04]
Train MSE: 4683203783.508417
Test  MSE: 4908476947.002458


***4 Part 2: Classification Task (Breast Cancer)***

***4.1 Task 1: Load and Split Dataset***

In [28]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("yasserh/breast-cancer-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'breast-cancer-dataset' dataset.
Path to dataset files: /kaggle/input/breast-cancer-dataset


In [29]:
import pandas as pd
import numpy as np

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder, StandardScaler

In [31]:
# Load dataset from the path downloaded by kagglehub:
df = pd.read_csv(path + "/breast-cancer.csv")
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [32]:
# Separate features (X) and target (Y)
# the correct target column name in this dataset is 'median_house_value':
X = df.drop('diagnosis', axis=1)
y = df['diagnosis']

In [33]:
# Feature scaling
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [34]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42 )

***4.2 Task 2: Complete all the Task***

***• Classification Task (Diabetes):***

Step 1: Baseline Model (No Regularization) Build a Logistic Regression model without
specifying any

1.   List item
2.   List item

regularization (default settings).

In [35]:
log_reg = LogisticRegression(max_iter=5000)
log_reg.fit(X_train, y_train)

LogisticRegression(max_iter=5000)


**LogisticRegression(max_iter=5000)**

***In a Jupyter environment, please rerun this cell to show the HTML representation or trust the notebook.***

***On GitHub, the HTML representation is unable to render, please try loading this page with nbviewer.org.***

In [36]:
log_reg.coef_

array([[ 4.41470952e-10, -9.90739776e-03, -1.80569429e-02,
        -5.93877477e-02, -1.71133973e-02, -1.06687155e-04,
        -1.66538334e-05,  9.53940445e-05,  4.46453855e-05,
        -2.02172454e-04, -8.17804040e-05, -6.17338761e-05,
        -1.31350048e-03, -9.79085421e-05,  2.66714451e-02,
        -7.79204511e-06, -7.75379435e-06, -5.03687770e-06,
        -3.54003090e-06, -2.17295912e-05, -3.48653062e-06,
        -1.05694403e-02, -2.39209316e-02, -6.26932225e-02,
         2.84854511e-02, -1.43990853e-04, -1.25080205e-05,
         1.28309254e-04,  2.32276941e-05, -2.99598218e-04,
        -9.20759207e-05]])

In [37]:
train_acc = accuracy_score(y_train, log_reg.predict(X_train))
test_acc = accuracy_score(y_test, log_reg.predict(X_test))

In [39]:
print("Baseline Training Accuracy:", train_acc)
print("Baseline Test Accuracy:", test_acc)

Baseline Training Accuracy: 0.9120879120879121
Baseline Test Accuracy: 0.956140350877193


Step 2: Hyperparameter Tuning Use GridSearchCV or RandomizedSearchCV to tune
hyperparameters for logistic regression models with regularization.  

In [40]:
param_grid = {
    "C": [0.01, 0.1, 1, 10, 100],
    "penalty": ["l1", "l2"],
    "solver": ["liblinear"]
}

In [41]:
grid = GridSearchCV(
    LogisticRegression(max_iter=5000),
    param_grid,
    cv=5,
    scoring="accuracy"
)

grid.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=LogisticRegression(max_iter=5000),
             param_grid={'C': [0.01, 0.1, 1, 10, 100], 'penalty': ['l1', 'l2'],
                         'solver': ['liblinear']},
             scoring='accuracy')

***In a Jupyter environment, please rerun this cell to show the HTML representation or trust the notebook.***

***On GitHub, the HTML representation is unable to render, please try loading this page with nbviewer.org.***

In [42]:
print("Best Parameters:", grid.best_params_)

Best Parameters: {'C': 100, 'penalty': 'l1', 'solver': 'liblinear'}


In [43]:
best_model = grid.best_estimator_

train_acc = accuracy_score(y_train, best_model.predict(X_train))
test_acc = accuracy_score(y_test, best_model.predict(X_test))

In [44]:
print("Tuned Training Accuracy:", train_acc)
print("Tuned Test Accuracy:", test_acc)

Tuned Training Accuracy: 0.9868131868131869
Tuned Test Accuracy: 0.9824561403508771


Step 3: Regularization Experiments (L1 vs L2) Train separate logistic regression models
using L1 (Lasso-like) and L2 (Ridge-like) regularization with the optimal hyperparameters.

In [49]:
param_grid = {
    'C': [0.001, 0.01, 0.1, 1, 10],
    'penalty': ['l1', 'l2']
}

In [50]:
log_l1 = LogisticRegression(
    penalty="l1",
    C=grid.best_params_["C"],
    solver="liblinear",
    max_iter=5000
)

log_l1.fit(X_train, y_train)

LogisticRegression(C=100, max_iter=5000, penalty='l1', solver='liblinear')

**LogisticRegression(C=100, max_iter=5000, penalty='l1', solver='liblinear')**

***In a Jupyter environment, please rerun this cell to show the HTML representation or trust the notebook.***

***On GitHub, the HTML representation is unable to render, please try loading this page with nbviewer.org.***

In [46]:
# L2 Regularization:
log_l2 = LogisticRegression(
    penalty="l2",
    C=grid.best_params_["C"],
    solver="liblinear",
    max_iter=5000
)

log_l2.fit(X_train, y_train)

LogisticRegression(C=100, max_iter=5000, solver='liblinear')

**LogisticRegression(C=100, max_iter=5000, solver='liblinear')**

***In a Jupyter environment, please rerun this cell to show the HTML representation or trust the notebook.***

***On GitHub, the HTML representation is unable to render, please try loading this page with nbviewer.org.***

In [47]:
print("L1 Coefficients:\n", log_l1.coef_)
print("\nL2 Coefficients:\n", log_l2.coef_)

L1 Coefficients:
 [[ 4.66027094e-09 -6.17664707e-01  1.15556222e-01 -9.17632005e-02
   1.02576425e-03  0.00000000e+00 -5.69503342e+01  1.43328412e+00
   1.66748531e+02 -2.20874509e+01  0.00000000e+00  3.69564192e+00
  -1.55177284e+00 -4.15706356e-01  1.90413237e-01  0.00000000e+00
  -1.65058851e+00 -7.17534158e+01  0.00000000e+00  0.00000000e+00
   0.00000000e+00 -2.11825837e-01  4.16578305e-01 -4.07367919e-02
   1.76112322e-02  2.90557853e+00 -8.43342122e+00  2.24312070e+01
   2.20628191e+01  2.69259103e+01  0.00000000e+00]]

L2 Coefficients:
 [[-2.12715847e-10 -4.73698845e-03 -8.57667361e-03 -2.84437840e-02
  -1.52333525e-02 -5.04849437e-05 -7.50239523e-06  4.45227214e-05
   2.14728348e-05 -9.57791799e-05 -3.91612506e-05 -3.09244197e-05
  -6.78907941e-04 -8.78248720e-05  1.11646787e-02 -4.10123930e-06
  -4.91532105e-06 -4.10095607e-06 -2.16000883e-06 -1.11155543e-05
  -1.87345006e-06 -4.85404239e-03 -1.09821084e-02 -2.87784619e-02
   1.85671621e-02 -6.64236049e-05  6.50727516e-06  7.

In [48]:
print("L1 Train Accuracy:", accuracy_score(y_train, log_l1.predict(X_train)))
print("L1 Test Accuracy:", accuracy_score(y_test, log_l1.predict(X_test)))

print("L2 Train Accuracy:", accuracy_score(y_train, log_l2.predict(X_train)))
print("L2 Test Accuracy:", accuracy_score(y_test, log_l2.predict(X_test)))

L1 Train Accuracy: 0.9868131868131869
L1 Test Accuracy: 0.9824561403508771
L2 Train Accuracy: 0.9098901098901099
L2 Test Accuracy: 0.956140350877193
